In [22]:
import os
import sys
import mlflow
from mlflow.tracking import MlflowClient

sys.path.insert(0, os.path.abspath("../../src"))
from utils.notify import notify_run_complete, notify_model_registered, notify_stage_change

_db_path = os.path.abspath("../../mlruns/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
mlflow.set_experiment("employee-attrition")

SLACK_URL = os.getenv("SLACK_WEBHOOK_URL", "")  # paste your Slack Incoming Webhook URL here
MODEL_NAME = "employee-attrition"


In [23]:
# client = MlflowClient()

In [24]:
# Register the best model (XGBoost had the highest AUC) into MLflow Model Registry
client = MlflowClient()

# xgb_run_id  = "ffceef00f9de4a0d97011e35cbb90631"
# xgb_run_id  = "b5ea866e0e854c1a9418281fc3cafd90"
xgb_run_id  = "6435f814834a4eadb2f86f0002089e21"

registered = mlflow.register_model(
    model_uri=f"runs:/{xgb_run_id}/model",
    name=MODEL_NAME,
)
print(f"Registered '{MODEL_NAME}' version {registered.version} from run {xgb_run_id}")

Registered model 'employee-attrition' already exists. Creating a new version of this model...
2026/06/13 14:49:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 6435f814834a4eadb2f86f0002089e21 has no artifacts at artifact path 'model', registering model based on models:/m-706b43ea5ef44a43a5a7a50583f590ea instead


Registered 'employee-attrition' version 6 from run 6435f814834a4eadb2f86f0002089e21


Created version '6' of model 'employee-attrition'.


In [25]:
# Add descriptions and required tags after registration
client.update_registered_model(
    name=MODEL_NAME,
    description=(
        "XGBoost classifier predicting employee attrition (Left / Stayed). "
        "Trained on 74 498 employees with 22 features retrieved from the Feast feature store. "
        "Input: label-encoded employee profile features. Output: binary label (0 = Stayed, 1 = Left)."
    ),
)

client.update_model_version(
    name=MODEL_NAME,
    version=registered.version,
    description=(
        f"Version {registered.version} — XGBoost (GridSearchCV, roc_auc scoring). "
        "Best params selected from grid: n_estimators, max_depth, learning_rate, subsample, colsample_bytree. "
        "Metrics: accuracy=0.761, f1=0.772, precision=0.773, recall=0.771."
    ),
)

# Required tags checked by the testing pipeline
client.set_model_version_tag(MODEL_NAME, registered.version, "team", "data-science")
client.set_model_version_tag(MODEL_NAME, registered.version, "use_case", "employee-attrition")
client.set_model_version_tag(MODEL_NAME, registered.version, "data_version", "v1")

print(f"Descriptions and tags added to '{MODEL_NAME}' version {registered.version}")

Descriptions and tags added to 'employee-attrition' version 6


In [26]:
# Notify Slack — training complete + model registered

xgb_metrics = {'accuracy': 0.7606711409395973, 'f1_score': 0.771878198567042, 'precision': 0.7728670253651038, 'recall': 0.7708918987988755}

notify_run_complete(
    run_name="xgboost-tree",
    metrics=xgb_metrics,
    run_id=xgb_run_id,
    webhook_url=SLACK_URL,
)
notify_model_registered(
    model_name=MODEL_NAME,
    version=registered.version,
    run_name="xgboost-tree",
    webhook_url=SLACK_URL,
)


[notify] Slack notification sent (status 200)
[notify] Slack notification sent (status 200)


200

In [27]:
# Promote model to Staging — run this cell when ready to test
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Staging",
)
notify_stage_change(
    model_name=MODEL_NAME,
    version=registered.version,
    old_stage="None",
    new_stage="Staging",
    webhook_url=SLACK_URL,
)
print(f"'{MODEL_NAME}' v{registered.version} → Staging")

# Promote to Production — uncomment when validated in Staging
# client.transition_model_version_stage(name=MODEL_NAME, version=registered.version, stage="Production")
# notify_stage_change(MODEL_NAME, registered.version, "Staging", "Production", webhook_url=SLACK_URL)
# print(f"'{MODEL_NAME}' v{registered.version} → Production")


[notify] Slack notification sent (status 200)
'employee-attrition' v6 → Staging


C:\Users\aarun\AppData\Local\Temp\ipykernel_30132\4203755610.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
